# Classification de texte : un classifieur Naive Bayes from scratch

**Série NLP — carnet 10.** Prolonge [02 · N-grammes](02_NGrammes_Modeles_De_Langue.ipynb) (estimation par MLE + lissage) et [06 · Recherche d'information](06-Recherche-Information-TFIDF-BM25.ipynb) (le sac-de-mots).

**Navigation** : [<< Précédent — 09. fastText (subwords)](09-FastText-Subword-Vectors.ipynb) | [Index de la série](README.md) | **10 (ce notebook)**

Les carnets précédents demandent « quelle est la probabilité de ce texte ? » (02) ou « quel document est le plus pertinent pour cette requête ? » (06). Ici on demande « **quelle est la catégorie de ce document ?** ». Le **Naive Bayes** répond en factorisant la probabilité d'un document par sa catégorie grâce à l'hypothèse d'indépendance « naïve » des mots :

$$P(c \mid d) \;\propto\; P(c)\prod_{w\in d} P(w \mid c)$$

Tout est **mesuré** (précision out-of-sample sur un corpus auto-contenu, aucun téléchargement) et le scoreur from scratch est **confronté à `sklearn`** — la machine réelle — sur le même split, avec le même vocabulaire d'apprentissage et la même tokenisation.


In [1]:
import re
import math
import statistics

# --- Corpus auto-contenu : 4 categories x 8 documents courts (aucun telechargement) ---
CORPUS = {
    "sport": [
        "l'equipe du stade remporte le match sur un dernier but",
        "l'arbitre siffle la fin du match dans le stade",
        "le coach motive l'equipe avant le match decisif",
        "le gardien arrete le penalty au dernier moment",
        "les joueurs celebrent la victoire dans le stade",
        "le buteur marque le but decisif du match",
        "le score final du match est historique au stade",
        "l'equipe entree au stade pour le match de la saison",
    ],
    "tech": [
        "le developp corrige un bug dans le logiciel",
        "le serveur renvoie une erreur dans le log du systeme",
        "la puce integre un accelerateur dans le processeur",
        "le framework compile le code en quelques secondes",
        "l'application consomme la memoire au demarrage du systeme",
        "l'algorithme accelere le calcul dans le logiciel",
        "l'interface du logiciel interroge le serveur du systeme",
        "le code du systeme compile sans erreur",
    ],
    "science": [
        "l'experience mesure la constante avec une grande precision",
        "le paper publie les resultats de l'observation",
        "l'hypothese est confirmee par les donnees recueillies",
        "le chercheur analyse les echantillons au microscope",
        "la theorie predit le comportement du systeme physique",
        "l'article rapporte une mesure de l'energie",
        "les mesures de l'experience valident le modele theorique",
        "l'observation confirme le resultat de l'experience",
    ],
    "finance": [
        "la bourse enregistre une hausse des indices ce matin",
        "la banque annonce un resultat net en hausse",
        "l'action cote plus haut a la fin de la seance",
        "l'investisseur choisit l'obligation pour le credit",
        "le taux d'interet guide la decision d'emprunt",
        "le portefeuille subit une baisse des valeurs",
        "le dividende est verse aux actionnaires du marche",
        "le marche anticipe une revision des taux",
    ],
}
CATEGORIES = list(CORPUS)
print("corpus :", {c: len(CORPUS[c]) for c in CATEGORIES})
print("total  :", sum(len(v) for v in CORPUS.values()), "documents (auto-contenu, reproductible)")

corpus : {'sport': 8, 'tech': 8, 'science': 8, 'finance': 8}
total  : 32 documents (auto-contenu, reproductible)


## 1. Le sac-de-mots — tokenisation from scratch

Comme le carnet 06, on réduit chaque document à un **sac de mots** (multiset). Tokenisation minimale : minuscules, mots = suites d'alphabétiques. Aucun outil externe.

In [2]:
def tokeniser(texte):
    """Minuscules, suites de lettres ; même token_pattern que le témoin sklearn."""
    return re.findall(r"[a-z]+", texte.lower())

# Le vocabulaire et le split n'utilisent que le train : aucun mot du test
# n'entre dans le modèle. Les mots OOV seront ignorés, comme CountVectorizer.
train_docs, test_docs = {}, {}
for c in CATEGORIES:
    train_docs[c] = CORPUS[c][:6]
    test_docs[c]  = CORPUS[c][6:]
VOCAB = sorted({w for docs in train_docs.values() for t in docs for w in tokeniser(t)})
print(f"vocabulaire du train : {len(VOCAB)} mots")
print("exemples :", VOCAB[:12])


vocabulaire du train : 118 mots
exemples : ['a', 'accelerateur', 'accelere', 'action', 'algorithme', 'analyse', 'annonce', 'application', 'arbitre', 'arrete', 'article', 'au']


## 2. L'estimateur Naive Bayes — MLE + lissage, en log-espace

L'hypothèse « naïve » factorise la vraisemblance. On estime par **maximum de vraisemblance** (comptages) avec **lissage de Laplace (add-α)** pour qu'aucun mot absent du train ne fasse s'effondrer le produit à 0 :

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\text{count}(\ast, c) + \alpha\,|V|}$$

Le scoreur somme les **log-probabilités** (stabilité numérique) : $\log P(c) + \sum_w \log P(w \mid c)$. C'est exactement la tradition du carnet 02 (MLE + lissage), appliquée aux catégories au lieu des n-grammes.

In [3]:
def entrainer_nb(corpus, vocab, alpha=1.0):
    """Estime P(c) et P(w|c) par MLE + lissage de Laplace (add-alpha).

    Renvoie (log_priors, log_word_probs) : log_priors[c] = log P(c), et
    log_word_probs[c] est un dict mot -> log P(w|c).
    """
    V = len(vocab)
    n_total = sum(len(d) for d in corpus.values())
    log_priors, log_word_probs = {}, {}
    for c, docs in corpus.items():
        total_c = sum(len(tokeniser(t)) for t in docs)          # count(*, c)
        log_priors[c] = math.log(len(docs) / n_total)
        counts = {}
        for t in docs:
            for w in tokeniser(t):
                counts[w] = counts.get(w, 0) + 1
        def _lp(n):
            # log(0) = -inf (mot absent de la categorie au train) : le « trou »
            # que le lissage alpha > 0 vient combler.
            return math.log(n / (total_c + alpha * V)) if n > 0 else float("-inf")
        log_word_probs[c] = {w: _lp(counts.get(w, 0) + alpha) for w in vocab}
    return log_priors, log_word_probs

def score_nb(doc, log_priors, log_word_probs):
    """log P(c) + sum_w log P(w|c), pour chaque categorie c."""
    out = {}
    for c in log_priors:
        s = log_priors[c]
        for w in tokeniser(doc):
            if w in log_word_probs[c]:
                s += log_word_probs[c][w]   # OOV ignoré : même convention que CountVectorizer
        out[c] = s
    return out

def predire_nb(doc, log_priors, log_word_probs):
    """(categorie argmax, scores ; None si toutes les catégories sont à -inf)."""
    scores = score_nb(doc, log_priors, log_word_probs)
    finite = {c: s for c, s in scores.items() if math.isfinite(s)}
    return (max(finite, key=finite.get) if finite else None), scores

log_priors, log_word_probs = entrainer_nb(train_docs, VOCAB)
print(f"train : {sum(len(v) for v in train_docs.values())} docs | "
      f"test  : {sum(len(v) for v in test_docs.values())} docs (out-of-sample)")

train : 24 docs | test  : 8 docs (out-of-sample)


In [4]:
# --- Precision out-of-sample MESUREE (le resultat) ---
bonnes, total = 0, 0
for c in CATEGORIES:
    for t in test_docs[c]:
        pred, _ = predire_nb(t, log_priors, log_word_probs)
        bonnes += (pred == c)
        total += 1
        marque = "ok" if pred == c else "XX"
        print(f"  [{marque}] vrai={c:8} pred={str(pred):8}  {t[:42]}")
print(f"\nPRECISION OUT-OF-SAMPLE (from scratch) : {bonnes}/{total} = {bonnes/total:.2%}")


  [ok] vrai=sport    pred=sport     le score final du match est historique au 
  [ok] vrai=sport    pred=sport     l'equipe entree au stade pour le match de 
  [ok] vrai=tech     pred=tech      l'interface du logiciel interroge le serve
  [ok] vrai=tech     pred=tech      le code du systeme compile sans erreur
  [ok] vrai=science  pred=science   les mesures de l'experience valident le mo
  [ok] vrai=science  pred=science   l'observation confirme le resultat de l'ex
  [XX] vrai=finance  pred=sport     le dividende est verse aux actionnaires du
  [ok] vrai=finance  pred=finance   le marche anticipe une revision des taux

PRECISION OUT-OF-SAMPLE (from scratch) : 7/8 = 87.50%


### Lecture du résultat

Sept documents sur huit sont bien classés. L'erreur n'est pas un défaut d'implémentation : c'est ce que le modèle **pouvait** voir.

Le document mal classé, « le dividende est verse aux actionnaires du marche », se découpe en huit mots dont **cinq sont hors vocabulaire** — `dividende`, `verse`, `aux`, `actionnaires`, `marche`. Trois d'entre eux portent le sujet financier (`dividende`, `actionnaires`, `marche`) ; `verse` est le verbe et `aux` une préposition, absents du train par rareté et non par topicalité. Il ne reste au scoreur que trois mots connus du train :

| mot | sport | tech | science | finance |
|---|---|---|---|---|
| `le` | 9 | 9 | 3 | 3 |
| `est` | 0 | 0 | 1 | 0 |
| `du` | 3 | 2 | 1 | 0 |

Aucun ne penche vers `finance` : deux sur trois la contredisent, le troisième est absent de sa classe. Le classifieur a donc été privé de toute preuve utile, et la taille du vocabulaire du train — 118 mots pour 24 documents — dit pourquoi. `score_nb` n'additionne que les mots vus au train : ignorer les mots inconnus est nécessaire pour ne pas planter, mais transforme ici une absence d'information en information neutre.

Le second document finance est classé correctement alors qu'il perd lui aussi des mots — `marche`, `anticipe` et `revision` sont hors vocabulaire. Sa bonne classification ne doit donc rien au mot qui nomme le sujet : elle tient à `taux`, le seul mot financier que le train connaisse. Une lecture qui attribuerait le succès à `marche` se tromperait de preuve, exactement comme la précédente.

C'est la leçon transposable : une erreur de prédiction se diagnostique en regardant **quels mots ont réellement été comptés**, pas en soupçonnant le calcul. Un compte de comptages se lit avant un compte de scores.

## 3. Cross-check SOTA — la machine from scratch contre `sklearn`

Le scoreur **from scratch** est confronté à `CountVectorizer` + `MultinomialNB` sur le même split. Le vocabulaire est construit sur le train seul, les mots OOV sont ignorés, et le témoin reçoit `token_pattern=r"[a-z]+"` : il conserve ainsi les mots d'une lettre, exactement comme `tokeniser`. Les différences de conventions sont donc explicitement contrôlées.


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

X_train = [t for c in CATEGORIES for t in train_docs[c]]
y_train = [c for c in CATEGORIES for t in train_docs[c]]
X_test  = [t for c in CATEGORIES for t in test_docs[c]]
y_test  = [c for c in CATEGORIES for t in test_docs[c]]

vec = CountVectorizer(token_pattern=r"[a-z]+")
clf = MultinomialNB().fit(vec.fit_transform(X_train), y_train)
sk_pred = list(clf.predict(vec.transform(X_test)))
fs_pred = [predire_nb(t, log_priors, log_word_probs)[0] for t in X_test]

sk_bonnes = sum(p == t for p, t in zip(sk_pred, y_test))
fs_bonnes = sum(p == t for p, t in zip(fs_pred, y_test))
accorde   = sum(a == b for a, b in zip(fs_pred, sk_pred))
print(f"precision OOS : from scratch {fs_bonnes}/{len(y_test)}  |  sklearn {sk_bonnes}/{len(y_test)}")
print(f"accord from-scratch <-> sklearn sur le test : {accorde}/{len(y_test)}")
print("-" * 70)
for t, ytrue, pf, ps in zip(X_test, y_test, fs_pred, sk_pred):
    print(f"  vrai={ytrue:8} from-scratch={pf:8} sklearn={ps:8}  {t[:34]}")

precision OOS : from scratch 7/8  |  sklearn 7/8
accord from-scratch <-> sklearn sur le test : 8/8
----------------------------------------------------------------------
  vrai=sport    from-scratch=sport    sklearn=sport     le score final du match est histor
  vrai=sport    from-scratch=sport    sklearn=sport     l'equipe entree au stade pour le m
  vrai=tech     from-scratch=tech     sklearn=tech      l'interface du logiciel interroge 
  vrai=tech     from-scratch=tech     sklearn=tech      le code du systeme compile sans er
  vrai=science  from-scratch=science  sklearn=science   les mesures de l'experience valide
  vrai=science  from-scratch=science  sklearn=science   l'observation confirme le resultat
  vrai=finance  from-scratch=sport    sklearn=sport     le dividende est verse aux actionn
  vrai=finance  from-scratch=finance  sklearn=finance   le marche anticipe une revision de


### Lecture du résultat

Deux implémentations écrites indépendamment donnent le **même compte** (7/8) et, plus parlant, le **même verdict sur chacun des huit documents** — y compris le même désaccord avec la vérité (`le dividende ...` classé `sport` des deux côtés).

Il faut lire ce que cet accord prouve et ce qu'il ne prouve pas. Il prouve que la machine from scratch **calcule ce qu'elle prétend calculer** : même tokenisation, même vocabulaire construit sur le train, même ignorance des mots hors vocabulaire, même estimateur. C'est une validation de **code**.

Il ne prouve rien sur le **modèle**. Un témoin qui reproduit une erreur la confirme au lieu de l'absoudre : ici le désaccord avec la vérité n'est pas un bruit d'implémentation qu'un meilleur code effacerait, c'est la même absence d'information qui frappe les deux, parce qu'elle est dans les données — un vocabulaire de train trop petit pour couvrir le document — et non dans l'algorithme.

Un cross-check SOTA répond donc à « est-ce que j'ai bien codé ? », jamais à « est-ce que le modèle est bon ? ». La seconde question demande un autre instrument : plus de données, ou une mesure hors de ce test.

## 4. Ce que le lissage change — l'effet de α, mesuré

Sans lissage (`α = 0`), un mot connu du vocabulaire mais absent d'une catégorie rend son score `-inf`. Si toutes les catégories sont à `-inf`, le document est **indéterminé**, pas classé par l'ordre du dictionnaire. Pour α > 0, la table donne la précision sur 8 documents : sa résolution est donc de 12,5 points, insuffisante pour distinguer un vrai plateau entre plusieurs valeurs proches.


In [6]:
all_test = [(t, c) for c in CATEGORIES for t in test_docs[c]]
print("effet du lissage alpha : indéterminés si toutes les classes sont à -inf")
print(f"{'alpha':>6}  {'indetermines':>14}  {'classes':>8}  {'marge mediane':>14}  {'precision OOS':>18}")
for alpha in (0.0, 0.1, 0.5, 1.0, 2.0, 5.0):
    lp, lwp = entrainer_nb(train_docs, VOCAB, alpha=alpha)
    sorties = [predire_nb(t, lp, lwp) for t, tr in all_test]
    predictions = [p for p, _ in sorties]
    indetermines = sum(p is None for p in predictions)
    classes = len(all_test) - indetermines
    # Marge = écart entre la 1re et la 2e catégorie, en log-score. Elle n'existe
    # que si DEUX catégories au moins gardent un score fini : à alpha = 0 un seul
    # score survit par document classé, et la décision n'est plus un arbitrage.
    marges = []
    for _, scores in sorties:
        finis = sorted((s for s in scores.values() if math.isfinite(s)), reverse=True)
        if len(finis) >= 2:
            marges.append(finis[0] - finis[1])
    marge = f"{statistics.median(marges):.2f}" if marges else "-"
    b = sum(p == tr for p, (t, tr) in zip(predictions, all_test))
    print(f"{alpha:>6}  {indetermines:>14}  {classes:>8}  {marge:>14}  {b}/{len(all_test)}  =  {b/len(all_test):.2%}")


effet du lissage alpha : indéterminés si toutes les classes sont à -inf
 alpha    indetermines   classes   marge mediane       precision OOS
   0.0               3         5               -  4/8  =  50.00%
   0.1               0         8            6.33  7/8  =  87.50%
   0.5               0         8            3.39  7/8  =  87.50%
   1.0               0         8            2.15  7/8  =  87.50%
   2.0               0         8            1.36  7/8  =  87.50%
   5.0               0         8            0.59  7/8  =  87.50%


### Lecture du résultat

`α = 0` classe 5 documents sur 8 — 4 justes et 1 faux — et déclare les 3 autres **indéterminés** — pas « mal classés », indéterminés : aucune catégorie n'échappe au `-inf`. Il suffit d'un seul mot du document absent d'une classe pour la couler, puisque la somme de log-probabilités est contaminée par un seul `-inf`. Les trois documents qui tombent sont les deux `sport` et un `science`, ceux dont chaque catégorie ignore au moins un mot.

La colonne `marge mediane` affiche `-` à `α = 0`, et ce n'est pas une case manquante : aucun des cinq documents classés n'y garde deux scores finis, donc il n'y a aucun écart à mesurer. La décision n'y est plus un arbitrage entre catégories, mais la survie d'une seule — ce que le compteur `classes` sépare de la précision, qui ne dit que le nombre de bonnes réponses.

Puis la précision vaut 7/8 pour **toutes** les valeurs testées de 0,1 à 5,0. Il serait tentant d'en conclure que `α` n'a pas d'importance. La marge de décision dit le contraire :

| α | écart médian entre la 1re et la 2e catégorie |
|---|---|
| 0,1 | 6,33 |
| 1,0 | 2,15 |
| 5,0 | 0,59 |

La précision est stable, la **confiance ne l'est pas** : elle est divisée par onze. À `α = 5`, le terme de lissage `α·|V| = 590` pèse plus de dix fois les jetons réellement observés dans une catégorie (48 à 54), et la marge médiane tombe sous 0,6 nat — un mot de plus ou de moins peut renverser le verdict.

Le plateau mesure donc la **granularité d'un test de huit documents** (douze points et demi par document), pas l'insensibilité du modèle à `α`. Deux régimes différents — l'évidence des mots à petit `α`, le dénominateur à grand `α` — peuvent rendre le même compte. Un chiffre de précision ne se lit jamais seul ; la marge dit ce qu'il coûte.

## 5. Synthèse

| Question | Carnet | Machine |
|---|---|---|
| Probabilité du texte ? | 02 | n-grammes (MLE + lissage) |
| Document le plus pertinent ? | 06 | BM25 / TF-IDF |
| **Catégorie du document ? (ici)** | **10** | **Naive Bayes (MLE + lissage)** |

Le Naive Bayes from scratch **atteint une précision OOS mesurée** et **coïncide avec `sklearn`** sur ce petit test. Les deux machines partagent ici le vocabulaire du train, l'ignorance des mots OOV et la tokenisation ; le carnet 02 donne la même leçon d'estimation pour les n-grammes. La précision est mesurée sur 8 documents, pas une estimation de performance générale.


## 6. Exercices

Trois exercices pour consolider. Chaque fonction est un **stub** : complétez le corps. Les cellules s'exécutent telles quelles (elles renvoient `None` tant qu'incomplètes) — aucune erreur volontaire.

In [7]:
# Exercice 1 — Le variant Bernoulli (presence / absence)
def entrainer_nb_bernoulli(corpus, vocab, alpha=1.0):
    """Naive Bayes Bernoulli : un mot compte UNE fois par document (presence),
    pas autant de fois qu'il apparait.

    Attendu : des structures de meme forme que entrainer_nb, mais les effectifs
    comptent la presence (0/1 par doc) et le denominateur porte |V| (pas count(*,c)).
    """
    # TODO : completer
    # Indice : iterer sur `set(tokeniser(t))` pour les effectifs, et remplacer
    #         total_c par la somme des presences (nb de mots distincts par doc).
    result = None  # TODO etudiant
    return result

print(entrainer_nb_bernoulli(train_docs, VOCAB))   # -> None tant que non complete

None


In [8]:
# Exercice 2 — Expliquer une prediction
def contributions_mots(doc, categorie, log_word_probs):
    """Pour chaque mot du document, sa contribution log P(w|categorie).

    Attendu : une liste de paires (mot, contribution) triee par contribution
    decroissante — les mots qui « tirent » le plus vers la categorie.
    """
    # TODO : completer
    # Indice : [ (w, log_word_probs[categorie].get(w, float('-inf')))
    #           for w in tokeniser(doc) ]  puis trier par la 2e composante, decroissant.
    result = None  # TODO etudiant
    return result

print(contributions_mots("le match au stade", "sport", log_word_probs))   # -> None

None


In [9]:
# Exercice 3 — Les mots vides
def precision_sans_mots_vides(mots_vides):
    """Retire `mots_vides` de chaque document, re-entraine, et renvoie la
    precision OOS from scratch — pour mesurer si filtrer les mots vides aide.

    Attendu : un float (la precision), comparable a celle de la section 2.
    """
    # TODO : completer
    # Indice : filtrer tokeniser() pour exclure mots_vides, reconstruire un
    #         corpus tokenise, re-appeler entrainer_nb/predire_nb sur le test.
    result = None  # TODO etudiant
    return result

print(precision_sans_mots_vides({"le", "la", "de", "du", "a", "des"}))   # -> None

None


## Références

- **McCallum, A. & Nigam, K. (1998).** « A Comparison of Event Models for Naive Bayes Text Classification », AAAI-98 Workshop on Learning for Text Categorization. — modèles multinomial et Bernoulli.
- **Rennie, J. D. M. (2000).** « Learning to Classify Text Documents using Word Frequencies », rapport AIM-2000-016. — utilisation des fréquences de mots pour la classification.
- **sklearn** — `MultinomialNB` / `BernoulliNB` (l'organe de référence du cross-check).

**Place dans la série** : `10` — la machine probabiliste de la série appliquée à la **classification** (après le n-gramme 02 et la recherche d'information 06).
